In [1]:
import tqdm
import torch
from load_dotenv import load_dotenv
import os
import json
import re

# Basic transformer libraries
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
from transformers import Mistral3ForConditionalGeneration, FineGrainedFP8Config

# Tools
import requests
from collections import defaultdict
from typing import Callable, Any
from dataclasses import dataclass

c:\Users\dubos\Documents\Development\Portfolio\Growth_RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Settings informations
Once the code is finished, the following cell should be the only one needing modification to interact with it.

In [44]:
model_id = "mistralai/Ministral-3-3B-Instruct-2512"

prompt_system = "You are a helpful librarian trying to recommand books based on a query by a user. " \
"Your answer includes 3 books, with a short summary of each, their type and the link to download the book. " \
"If no link is available, simply say link: Not found. Do not suggest other buying/downloading platforms"

prompt_user = "Can you recommend an adventuring book to me please ?"




## Loading .env data

In [3]:
load_dotenv()
GUT_HOST = os.getenv("GUT_HOST")

In [4]:
GUT_HOST

'https://project-gutenberg-books-api1.p.rapidapi.com'

## Tool Creation

In [5]:
def fetch_book_url(title: str):
    """
    Get the url of a given book.

    Args:
        title: The title of the book that is being looked up. 
    """
    endpoint = "/books"
    headers = {
        "X-RapidAPI-Key" : os.getenv("RAPIDAPI"),
        "X-RapidAPI-Host": "project-gutenberg-free-books-api1.p.rapidapi.com"
        }
    params = {
        "title" : title,

    }
    response = requests.get(GUT_HOST+endpoint, params=params, headers=headers)
    if response.status_code == 200:
        response = response.json()
        if (response.get("results") and len(response["results"]) > 0):
            result = response["results"][0]["formats"]["text/plain; charset=us-ascii"]
        else:
            result = 'No url was found in the Project Gutenberg'
    return result #["results"][0]["formats"]["text/plain; charset=us-ascii"]

tools = [fetch_book_url]

@dataclass
class Tool:
    name: str
    func: Callable
    description: str
    schema: dict

tool_registry = {
    'fetch_book_url' : Tool(
        name='fetch_book_url',
        func=fetch_book_url,
        description="""
            Get the url of a given book.
        
            Args:
                title: The title of the book that is being looked up. 
            """,
        schema={
            'type':'object',
            'properties':{
                'title':{'type': 'string'}
            },
            'required': ['title']
        }
    )
}





## Translation to mistral API

In [6]:
def tool_to_mistral_schema(tool: Tool) -> dict:
    return {
        "type": "function",
        "function": {
            "name": tool.name,
            "description": tool.description.strip(),
            "parameters": tool.schema
        }
    }
mistral_tools = [tool_to_mistral_schema(tool) for tool in tool_registry.values()]

## Tool call handling

In [7]:
def execute_tool_call(tool_name:str, tool_args:dict) -> Any:
    if tool_name not in tool_registry:
        raise ValueError(f" Unknown tool: {tool_name} no in the toolbox")

    tool = tool_registry[tool_name]
    try:
        result = tool.func(**tool_args)
        return result
    except TypeError as e:
        raise ValueError(f"Invalid arguments for {tool_name}: {e}")

## Prompt initiation

In [45]:
chat = [
    {"role" : "system", "content" : prompt_system},
    {"role" : "user", "content" : prompt_user}
]

## Model initiation

In [9]:
# model_id = "mistralai/Ministral-3-3B-Instruct-2512"
model = Mistral3ForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=FineGrainedFP8Config(dequantize=True)
)

# model = AutoModelForCausalLM.from_pretrained(model)
tokenizer = AutoTokenizer.from_pretrained(model_id, device_map="auto", dtype=torch.bfloat16)
   


Loading weights: 100%|██████████| 458/458 [00:07<00:00, 62.80it/s]
[transformers] The tokenizer you are loading from 'mistralai/Ministral-3-3B-Instruct-2512' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


In [46]:
# Quick evaluation runs
tokenized_prompt = tokenizer.apply_chat_template(chat, tools=mistral_tools, tokenize=True, add_generation_prompt=True, return_tensors="pt").to(model.device) 
# inputs = tokenizer(tokenized_prompt, return_tensors="pt")
outputs = model.generate(**tokenized_prompt, max_new_tokens=500)
#tool call
# tool_call = {"name" : "fetch_book_url", "arguments": {"title":}}
print(tokenizer.decode(outputs[0]))

[transformers] Both `max_new_tokens` (=500) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<s>[SYSTEM_PROMPT]You are a helpful librarian trying to recommand books based on a query by a user. Your answer includes 3 books, with a short summary of each, their type and the link to download the book. If no link is available, simply say link: Not found. Do not suggest other buying/downloading platforms[/SYSTEM_PROMPT][AVAILABLE_TOOLS][{"type": "function", "function": {"name": "fetch_book_url", "description": "Get the url of a given book.\n\n            Args:\n                title: The title of the book that is being looked up.", "parameters": {"type": "object", "properties": {"title": {"type": "string"}}, "required": ["title"]}}}][/AVAILABLE_TOOLS][INST]Can you recommend an adventuring book to me please ?[/INST][TOOL_CALLS]fetch_book_url[ARGS]{"title": "The Name of the Wind"}[TOOL_CALLS]fetch_book_url[ARGS]{"title": "The Lies of Locke Lamora"}[TOOL_CALLS]fetch_book_url[ARGS]{"title": "The City of Brass: A Dubai Mystery"}</s>


## Decoding the tool calls
Beware, Eval is used here, and could lead to security issues. May be needed to change it later on.

In [ ]:
def extract_assistant_response(response_text: str) -> tuple:
    """
    Extract the actual assistant response before tool calls.
    Returns: (assistant_text, tool_calls_text)
    """
    # Find where tool calls start
    tool_calls_start = response_text.find('[TOOL_CALLS]')
    
    if tool_calls_start == -1:
        # No tool calls—entire response is the answer
        return response_text, None
    
    # Extract text before tool calls
    assistant_text = response_text[:tool_calls_start].strip()
    tool_calls_text = response_text[tool_calls_start:]
    
    return assistant_text, tool_calls_text


def parse_tool_calls(tool_calls_texts: str) -> list:
    """Parse sequential Mistral tool calls"""
    tool_calls = []
    pattern = r'\[TOOL_CALLS\](.*?)\[ARGS\](.*?)(?=\[TOOL_CALLS\]|$)'
    matches = re.findall(pattern, tool_calls_texts, re.DOTALL)
    
    for tool_name, args_str in matches:
        tool_name = tool_name.strip()
        args_str = args_str.strip().rstrip('</s>')

        if not tool_name or not args_str:
            continue

        try:
            args = json.loads(args_str)
            tool_calls.append({
                "name": tool_name,
                "arguments": args
            })
        except json.JSONDecodeError:
            print(f"⚠ Failed to parse: {tool_name} | {args_str}")
    
    return tool_calls



Assistant: [TOOL_CALLS]fetch_book_url[ARGS]{"title": "The Name of the Wind"}[TOOL_CALLS]fetch_book_url[ARGS]{"title": "The Lies of Locke Lamora"}[TOOL_CALLS]fetch_book_url[ARGS]{"title": "The City of Brass: A Du
Tool calls found: 3
  - fetch_book_url: {'title': 'The Name of the Wind'}
  - fetch_book_url: {'title': 'The Lies of Locke Lamora'}
  - fetch_book_url: {'title': 'The City of Brass: A Dubai Mystery'}


In [47]:
# Now use it:
decoded = tokenizer.decode(outputs[0])
assistant_text, tool_calls_text = extract_assistant_response(decoded)
tool_calls = parse_tool_calls(tool_calls_text)

print(f"Assistant: {tool_calls_text[:200]}")
print(f"Tool calls found: {len(tool_calls)}")
for call in tool_calls:
    print(f"  - {call['name']}: {call['arguments']}")

Assistant: [TOOL_CALLS]fetch_book_url[ARGS]{"title": "The Name of the Wind"}[TOOL_CALLS]fetch_book_url[ARGS]{"title": "The Lies of Locke Lamora"}[TOOL_CALLS]fetch_book_url[ARGS]{"title": "The City of Brass: A Du
Tool calls found: 3
  - fetch_book_url: {'title': 'The Name of the Wind'}
  - fetch_book_url: {'title': 'The Lies of Locke Lamora'}
  - fetch_book_url: {'title': 'The City of Brass: A Dubai Mystery'}


In [37]:
# tool_call_list = tokenizer.decode(outputs[0]).split("[TOOL_CALLS]")
# call_dict = defaultdict(list)

# for call in tool_call_list[1:]:
#     args = call.split("[ARGS]")
#     # call_dict = {}
#     call_dict[args[0]].append(eval(args[1].rstrip('</s>')))
# call_dict

In [48]:
tool_call_results = []

for call in tool_calls:
    result = execute_tool_call(call["name"], call["arguments"])
    tool_call_results.append({
        "tool": call["name"],
        "arguments": call["arguments"],
        "result": result
        })
    print(f"\n{call['name']} for {call['arguments']}: {result}")


fetch_book_url for {'title': 'The Name of the Wind'}: No url was found in the Project Gutenberg

fetch_book_url for {'title': 'The Lies of Locke Lamora'}: No url was found in the Project Gutenberg

fetch_book_url for {'title': 'The City of Brass: A Dubai Mystery'}: No url was found in the Project Gutenberg


In [39]:
# tool_call_summary = []
# tool_call_result = {}
# for key in call_dict.keys():
#     for value in call_dict[key]:
#         # print(key, value)
#         tool_name = 
#         args = dict(value)
#         tool_call = {"name": key, "arguments": value}
#         tool_call_summary.append(tool_call)
#         result = execute_tool_call(key, args)
#         tool_call_result[args.values()] = result
#         print(result)
# tool_call_result

## Feeding tool output back to the LLM

In [40]:
# tokenizer.decode(outputs[0])

In [49]:
chat.append({"role":"assistant", "content": tool_calls_text})
chat.append({"role": "tool", "content": json.dumps(tool_call_results)})

In [50]:
chat

[{'role': 'system',
  'content': 'You are a helpful librarian trying to recommand books based on a query by a user. Your answer includes 3 books, with a short summary of each, their type and the link to download the book. If no link is available, simply say link: Not found. Do not suggest other buying/downloading platforms'},
 {'role': 'user',
  'content': 'Can you recommend an adventuring book to me please ?'},
 {'role': 'assistant',
  'content': '[TOOL_CALLS]fetch_book_url[ARGS]{"title": "The Name of the Wind"}[TOOL_CALLS]fetch_book_url[ARGS]{"title": "The Lies of Locke Lamora"}[TOOL_CALLS]fetch_book_url[ARGS]{"title": "The City of Brass: A Dubai Mystery"}</s>'},
 {'role': 'tool',
  'content': '[{"tool": "fetch_book_url", "arguments": {"title": "The Name of the Wind"}, "result": "No url was found in the Project Gutenberg"}, {"tool": "fetch_book_url", "arguments": {"title": "The Lies of Locke Lamora"}, "result": "No url was found in the Project Gutenberg"}, {"tool": "fetch_book_url", 

In [51]:
# Quick evaluation runs
tokenized_prompt = tokenizer.apply_chat_template(chat, tools=mistral_tools, tokenize=True, add_generation_prompt=True, return_tensors="pt").to(model.device) 
# inputs = tokenizer(tokenized_prompt, return_tensors="pt")
outputs = model.generate(**tokenized_prompt.to(model.device), max_new_tokens=500)
#tool call
# tool_call = {"name" : "fetch_book_url", "arguments": {"title":}}
print(tokenizer.decode(outputs[0]))

[transformers] Both `max_new_tokens` (=500) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<s>[SYSTEM_PROMPT]You are a helpful librarian trying to recommand books based on a query by a user. Your answer includes 3 books, with a short summary of each, their type and the link to download the book. If no link is available, simply say link: Not found. Do not suggest other buying/downloading platforms[/SYSTEM_PROMPT][AVAILABLE_TOOLS][{"type": "function", "function": {"name": "fetch_book_url", "description": "Get the url of a given book.\n\n            Args:\n                title: The title of the book that is being looked up.", "parameters": {"type": "object", "properties": {"title": {"type": "string"}}, "required": ["title"]}}}][/AVAILABLE_TOOLS][INST]Can you recommend an adventuring book to me please ?[/INST][TOOL_CALLS]fetch_book_url[ARGS]{"title": "The Name of the Wind"}[TOOL_CALLS]fetch_book_url[ARGS]{"title": "The Lies of Locke Lamora"}[TOOL_CALLS]fetch_book_url[ARGS]{"title": "The City of Brass: A Dubai Mystery"}</s></s>[TOOL_RESULTS][{"tool": "fetch_book_url", "arguments